# Week 10b — is the random network typical? Three more shuffles, then wrap up

## Week 10 (fresh seeds 93000–93049, readout C4)
Connectome **46/50** (pass). Random network (week 7's shuffle, seed 0) **50/50**. Pre-registered verdict: no significant difference on any milestone (success: 0 seeds only the connectome, 4 only the random network, p = 0.125). Placement favoured the random network: 5.4 vs 14.3 mm (p < 0.0001). That is **one** random network.

## This notebook (~2 h on a T4)
Three more shuffles (`random_network(full, seed=s)`, s = 1, 2, 3: every connection's target permuted; in/out-degrees, synapse counts and signs kept), each with exactly the rules seed 0 had, applied to its own wiring and activity:
- **w_scale** by week 7's activity match: the share of non-input neurons firing ≥ 1 Hz on week 7's 17 reference states equals the connectome's (batched grid search from the connectome's w_scale). Check first: the same search on seed 0 should land near week 7's 1.13.
- its own encoder channels, its own active neurons + all DNs as the pool, readout **C4**, DAgger 4 × 30 (seeds 64000+), test on **93000–93049**, feedback cut on 10.

The connectome's run is week 10's (attach week 10's output), so every network is tested on the same seeds.

## Rules (fixed before the run)
1. **Pass mark unchanged: ≥ 40/50**, for every network.
2. **Each random network vs the connectome:** week 7's verdict (paired by seed, success → carried to zone → lifted; the first exact McNemar p < 0.05 decides); placement and time compared only if both have ≥ 10 successes.
3. **Final, across all 4 random networks (seed 0 from week 10 + seeds 1–3):** *the wiring helps* if the connectome is significantly better than a majority (≥ 3 of 4); *the wiring hurts* if a majority is significantly better than the connectome; otherwise *no effect of the wiring shown*. Placement reported the same way as a secondary result.
4. Then the project is wrapped up, whatever the success counts.

Inputs (Kaggle inputs, or local folders; see the README's *Reproducing* section): this repo (attached as a dataset, or cloned by the setup cell), the data folder from `scripts/fetch_connectome.py`, **week 10's notebook output** (and week 9's, only if week 10's isn't available: then the connectome C4 is re-tested from week 9's data).

### Setup

In [ ]:
import sys, subprocess, os, shutil, zipfile, re
from pathlib import Path

# ---- where are we? -------------------------------------------------------------------------------
# Kaggle is checked first: its image also ships the google.colab package, but can't mount Drive.
KAGGLE_INPUT = Path('/kaggle/input')
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or KAGGLE_INPUT.is_dir()
IN_COLAB = not IN_KAGGLE and Path('/var/colab/hostname').exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
HOST = 'colab' if IN_COLAB else 'kaggle' if IN_KAGGLE else 'laptop'
if IN_COLAB or IN_KAGGLE:
    os.environ.setdefault('MUJOCO_GL', 'egl')                      # headless video; set before mujoco loads
    os.environ.setdefault('NUMBA_CACHE_DIR', '/tmp/numba_cache')   # the code folder is read-only on Kaggle

# ---- packages ------------------------------------------------------------------------------------
PKGS = ['brian2', 'numba', 'mujoco', 'imageio', 'imageio-ffmpeg']
if IN_COLAB or IN_KAGGLE:
    try:
        import brian2, numba, mujoco, imageio_ffmpeg
    except ImportError:
        if IN_KAGGLE:
            # A batch run (Save & Run All) can't survive a kernel restart, so numpy is pinned to the
            # installed version: nothing gets upgraded, nothing needs a restart.
            import numpy
            r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PKGS, f'numpy=={numpy.__version__}'])
            if r.returncode:
                raise RuntimeError("pip install failed. Notebook settings (right panel) -> Internet: On. "
                                   "Kaggle only offers Internet to phone-verified accounts.")
            import brian2, numba, mujoco, imageio_ffmpeg
        else:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PKGS], check=True)
            print("packages installed -> restarting runtime. When it reconnects, choose Runtime -> Run all again.")
            os.kill(os.getpid(), 9)

# ---- folders -------------------------------------------------------------------------------------
# Kaggle: the inputs are read-only and can come from several datasets and earlier notebook versions.
# They are linked into one tree with the repo layout (sim/, arm/, data/, mujoco_menagerie/), so every
# path below is the same as in a local checkout. Results go to /kaggle/working.
def _make_real(d):
    """Make folder d (under ROOT) a real folder. Any part of its path that is a link into a read-only input
    becomes a real folder holding links to that input's contents, so a second input can add to it."""
    p = ROOT
    p.mkdir(parents=True, exist_ok=True)
    for name in Path(d).relative_to(ROOT).parts:
        p = p / name
        if p.is_symlink() and p.is_dir():
            t = p.resolve(); p.unlink(); p.mkdir()
            for k in sorted(t.iterdir()):
                (p / k.name).symlink_to(k)
        elif not p.exists():
            p.mkdir()

def _link_tree(src, dst):
    """Link everything under src into dst, folder by folder, so several inputs can fill one tree."""
    _make_real(dst)
    for s in sorted(src.iterdir()):
        d = dst / s.name
        if d.is_symlink() and s.is_dir() and d.is_dir():   # a folder another input already linked: split it
            t = d.resolve(); d.unlink(); _link_tree(t, d)
        if s.is_dir() and d.is_dir() and not d.is_symlink():
            _link_tree(s, d)
        elif not d.exists() and not d.is_symlink():
            d.symlink_to(s)

_RUN = re.compile(r'week\d+[a-z]*(_[a-z]+)?')        # week7, week8b, week8b_quick: a notebook's results folder

def _find_roots(top, depth=4):
    """('tree', p): p holds sim/, arm/ or mujoco_menagerie/ (the repo, uploaded as a dataset).
       ('data', p): p is the data folder itself (weights_*/, wsyn_choice.json).
       ('runs', p): p holds week*/ folders (an earlier notebook version's output)."""
    found = []
    def walk(p, d):
        try:
            kids = sorted(p.iterdir())
        except OSError:
            return
        names = {k.name for k in kids}
        zips = [z for z in kids if z.suffix == '.zip' and z.stem in ('sim', 'arm', 'mujoco_menagerie')
                and z.stem not in names]           # in case an upload kept its folders zipped
        if zips:
            out = Path('/tmp/kaggle_unzip') / p.name
            for z in zips:
                if not (out / z.stem).exists():
                    zf = zipfile.ZipFile(z)
                    inside = all(n.startswith(z.stem + '/') for n in zf.namelist())
                    zf.extractall(out if inside else out / z.stem)
            found.append(('tree', out))
        if d > 0 and names & {'sim', 'arm', 'mujoco_menagerie'}:
            found.append(('tree', p))
        elif d > 0 and ('wsyn_choice.json' in names or any(n.startswith('weights_') for n in names)):
            found.append(('data', p))              # brain/data itself (an upload can drop the brain/ level)
        elif d > 0 and any(k.is_dir() and _RUN.fullmatch(k.name) for k in kids):
            found.append(('runs', p))
        elif d < depth:
            for k in kids:
                if k.is_dir():
                    walk(k, d + 1)
    walk(top, 0)
    return found

REPO_URL = 'https://github.com/Notlord69/fly-connectome-arm'   # used on Kaggle only if the repo isn't attached
if IN_KAGGLE:
    ROOT = Path('/tmp/fly-connectome-arm')                 # links only, rebuilt every session
    shutil.rmtree(ROOT, ignore_errors=True)
    ROOTS = _find_roots(KAGGLE_INPUT)
    for kind, p in sorted(ROOTS, key=lambda r: r[0] != 'runs'):   # notebook outputs first: newest results win
        if kind == 'tree':
            _link_tree(p, ROOT)
        elif kind == 'data':
            _link_tree(p, ROOT / 'data')
        else:
            for k in sorted(p.iterdir()):
                if k.is_dir() and _RUN.fullmatch(k.name):
                    _link_tree(k, ROOT / 'data' / k.name)
    print("Kaggle inputs:", *[f"{k:5s} {p}" for k, p in ROOTS], sep="\n  ")
    if not (ROOT / 'sim' / 'week10.py').exists():         # the code isn't attached: clone it
        src = Path('/tmp/fly-connectome-arm-src')
        if not src.exists():
            subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO_URL, str(src)], check=True)
        _link_tree(src, ROOT)
    CODE = ROOT
    if not (ROOT / 'mujoco_menagerie' / 'franka_emika_panda').exists():
        subprocess.run(['bash', str(CODE / 'scripts' / 'fetch_panda_model.sh'), '/tmp/mujoco_menagerie'], check=True)
        os.environ['FLYARM_MENAGERIE'] = '/tmp/mujoco_menagerie'
    DATA = ROOT / 'data'
elif IN_COLAB:
    CODE = Path('/content/drive/MyDrive/fly-connectome-arm')
    DATA = Path(os.environ.get('FLYARM_DATA_DIR', CODE / 'data'))
else:                                                       # a local checkout: this notebook is in notebooks/
    CODE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'sim' / 'week10.py').exists())
    DATA = Path(os.environ.get('FLYARM_DATA_DIR', CODE / 'data'))
os.environ['FLYARM_DATA_DIR'] = str(DATA)
RUNS = Path('/kaggle/working') if IN_KAGGLE else DATA      # where this notebook's results go
ARM = CODE / 'arm' if (CODE / 'arm' / 'panda_env.py').exists() else None
sys.path.insert(0, str(CODE))
if ARM: sys.path.insert(0, str(ARM))

def out_dir(name):
    """This notebook's results folder. On Kaggle it is /kaggle/working/<name>, saved with the version; if an
    earlier version's output is attached as an input, its <name>/ is copied in first, so RESUME carries on."""
    o = RUNS / name
    if RUNS != DATA and not o.exists() and (DATA / name).exists():
        shutil.copytree(DATA / name, o)
        print(f"resuming: copied the earlier run's {name}/ into {o}")
    o.mkdir(parents=True, exist_ok=True)
    return o


# ---- week 10b -----------------------------------------------------------------------------------
QUICK = False            # True: smoke test (DAgger 3 x 6, 10 test seeds, one shuffle), results in week10b_quick/
RESUME = True
SHUFFLES = (1, 2, 3)     # random_network seeds (week 7/10 used seed 0)
CHECK_SEED0 = True       # rerun week 7's activity match on seed 0 (~2 min): should give ~1.13
K_WIDE = 3000
USE_GPU = True
GPU_DTYPE = 'float32'
MAX_BATCH = 64

GPU = None
if USE_GPU:
    try:
        import torch
        if torch.cuda.is_available():
            GPU = f"{torch.cuda.get_device_name(0)} (x{torch.cuda.device_count()})"
            print(f"GPU: {GPU}; torch {torch.__version__}")
        else:
            print("USE_GPU = True but no GPU. Kaggle: Session options -> Accelerator -> GPU T4 x2.")
    except ImportError:
        print("torch is not installed here -> numba on the CPU")

for p in ['sim/batch7.py', 'sim/network7.py', 'sim/interface.py', 'sim/expert8.py', 'sim/pool8.py', 'sim/week9.py',
          'sim/week10.py', 'sim/week10b.py', 'sim/video7.py', 'data/weights_male-cns_v1.0/meta.json',
          'data/shortlist_week1.csv', 'data/wsyn_choice.json', 'data/week7/week7_random_network_meta.json']:
    print(f"{p:50s} {'ok' if ((CODE / p) if p.startswith('sim/') else (DATA / p.split('/', 1)[1])).exists() else 'MISSING'}")
print(f"{'week 4 arm code':50s} {ARM if ARM else 'MISSING'}")
print(f"{'week 10 output (data/week10/)':50s} {'found' if (DATA / 'week10' / 'week10_connectome__c4.pkl').exists() else 'NOT FOUND'}")

### Load the connectome, the neuron groups, the arm and week 10's runs

In [ ]:
import json, time, gc, pickle
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings('ignore', module='brian2')
from sim.lif import WeightSet, peak_mem_gb
from sim.pickplace import SIGNALS8
from sim.interface import Encoder
from sim.batch7 import make_engine, BatchBrain
from sim.network7 import wiring_labels, random_network, network_stats, degrees_equal
from sim.week7 import reference_states, comparison_rows
from sim.expert8 import expert8
from expert import expert as expert4
from sim.pool8 import active_neurons
from sim.week10 import TEST10, CUT10, week9_warm, run_network10
from sim.week10b import activity_target, match_shuffle, load_week10, compare10b, describe10b, restrict
from panda_env import PandaPickPlace, OBS_NAMES

T_START = time.perf_counter()
OUT = out_dir('week10b_quick' if QUICK else 'week10b')
ws = WeightSet(DATA / 'weights_male-cns_v1.0')
W = json.loads((DATA / 'wsyn_choice.json').read_text())['w_scale']
W_RAND0 = json.loads((DATA / 'week7' / 'week7_random_network_meta.json').read_text())['w_scale']
JO = ws.by_type(r'^JO-(C|E)', cls='cb_sensory')
shortlist = pd.read_csv(DATA / 'shortlist_week1.csv')
g2i = pd.Series(ws.neurons.index, index=ws.neurons.bodyId)
DN_cand = g2i.loc[shortlist.query("group == 'middle_descending'").bodyId].to_numpy()
ALL_DN = ws.idx(ws.neurons['cls'].astype(str).str.contains('descending', case=False))
full = ws.subnetwork(np.arange(ws.N))
ws._out = None; gc.collect()
env = PandaPickPlace()

N_ROUNDS = 3 if QUICK else 4
N_PER_ROUND = 6 if QUICK else 30
if QUICK:
    SHUFFLES = SHUFFLES[:1]
CFG = dict(
    train_seeds=[list(range(64_000 + 1000 * r, 64_000 + 1000 * r + N_PER_ROUND)) for r in range(N_ROUNDS)],  # = weeks 7-10
    betas=[1.0] + [0.0] * (N_ROUNDS - 1),
    test_seeds=TEST10[:10] if QUICK else TEST10,                                                         # = week 10
    cut_seeds=CUT10[:3] if QUICK else CUT10,
    k=K_WIDE, n_video=1, use_gpu=GPU is not None, dtype=GPU_DTYPE, max_batch=MAX_BATCH, cpu_batch=1)
print(f"male CNS: {ws.N:,} neurons; w_scale connectome {W}, random seed 0 {W_RAND0:.3f} (week 7); shuffles {SHUFFLES}; "
      f"test {CFG['test_seeds'][0]}-{CFG['test_seeds'][-1]}; results in {OUT}")

def wide_setup(sub, w_scale, label):
    # week 8b's rule: every non-input neuron firing >= 1 Hz during one expert episode, + all DNs
    keep = np.asarray(sub['keep'])
    enc = Encoder(JO, wiring_labels(sub, JO, 16), SIGNALS8, kind='wiring')
    eng = make_engine(sub, JO, GPU is not None, GPU_DTYPE)
    others = np.setdiff1d(np.arange(len(keep)), np.searchsorted(keep, JO))
    s_ep = reference_states(env, expert8, OBS_NAMES, n=60)
    t0 = time.perf_counter()
    act_local, _ = active_neurons(BatchBrain(eng, enc, ALL_DN), w_scale, s_ep, others)
    active = keep[act_local]
    print(f"{label}: {len(active):,} non-input neurons fire >= 1 Hz during an expert episode "
          f"({len(active) / len(others):.1%}); {np.isin(ALL_DN, active).sum()} of them are DNs  "
          f"[{time.perf_counter() - t0:.0f} s]")
    del eng; gc.collect()
    return np.union1d(active, ALL_DN)

W10 = load_week10(DATA / 'week10')
if 'connectome, C4' in W10:
    CONN = W10['connectome, C4']
    print(f"week 10's connectome C4: {CONN['summary']['success']}/{CONN['summary']['n']} on {CONN['test_seeds'][0]}-{CONN['test_seeds'][-1]}")
else:
    WARM = week9_warm(DATA / 'week9', K_WIDE)
    assert WARM, "attach week 10's notebook output (or week 9's): the connectome's run is needed for the comparison"
    print("week 10's output not attached: re-testing the connectome's C4 from week 9's data (~12 min)")
    CONN = run_network10('connectome, C4', full, JO, WARM['record'][WARM['pool_cols']], env, OBS_NAMES,
                         dict(CFG, test_seeds=TEST10, cut_seeds=CUT10), OUT, W, 'C4', resume=RESUME,
                         extra_record=DN_cand, warm=WARM)
CONN_T = restrict(CONN, CFG['test_seeds'])
RANDS = {}
if 'random network, C4' in W10:
    RANDS['random network s0, C4 (week 10)'] = restrict(W10['random network, C4'], CFG['test_seeds'])

## Step 1: the activity target, and week 7's match re-checked on seed 0 (~3 min)

In [ ]:
S_REF = reference_states(env, expert4, OBS_NAMES)          # week 7's matching input: week 4 expert, seed 99999, 17 steps
OTHERS = np.setdiff1d(np.arange(ws.N), JO)                 # non-input neurons (every network keeps all neurons)
enc_c = Encoder(JO, wiring_labels(full, JO, 16), SIGNALS8, kind='wiring')
eng_c = make_engine(full, JO, GPU is not None, GPU_DTYPE)
TARGET = activity_target(BatchBrain(eng_c, enc_c, ALL_DN), W, S_REF, OTHERS)
del eng_c, enc_c; gc.collect()
print(f"connectome at w_scale {W}: {TARGET:.2%} of non-input neurons fire >= 1 Hz (week 7: 4.5%)")
ST_FULL = network_stats(full)
MATCH = {}

def matched(s):
    rnet = random_network(full, seed=s)
    same = degrees_equal(ST_FULL, network_stats(rnet))
    enc = Encoder(JO, wiring_labels(rnet, JO, 16), SIGNALS8, kind='wiring')
    eng = make_engine(rnet, JO, GPU is not None, GPU_DTYPE)
    m = match_shuffle(BatchBrain(eng, enc, ALL_DN), W, TARGET, S_REF, OTHERS, OUT / f'week10b_shuffle{s}_match.json', RESUME)
    del eng, enc; gc.collect()
    m['degrees_equal'] = bool(same)
    print(f"shuffle {s}: every neuron keeps its in- and out-degree: {same}; w_scale {m['w_scale']:.3f} -> "
          f"{m['frac_firing']:.2%} firing (target {TARGET:.2%})")
    return rnet, m

if CHECK_SEED0:
    rnet0, MATCH[0] = matched(0)
    del rnet0; gc.collect()
    print(f"CHECK: seed 0 re-matched at {MATCH[0]['w_scale']:.3f}; week 7 used {W_RAND0:.3f} "
          f"({100 * (MATCH[0]['w_scale'] / W_RAND0 - 1):+.1f}%)")

## Step 2: three more random networks, readout C4 (~38 min each)

In [ ]:
for s in SHUFFLES:
    rnet, MATCH[s] = matched(s)
    pool = wide_setup(rnet, MATCH[s]['w_scale'], f'random network s{s}')
    name = f'random network s{s}, C4'
    RANDS[name] = run_network10(name, rnet, JO, pool, env, OBS_NAMES, CFG, OUT, MATCH[s]['w_scale'], 'C4',
                                resume=RESUME, extra_record=DN_cand)
    r = RANDS[name]['summary']
    print(f"\n{name.upper()}: {r['success']}/{r['n']} (95% CI {r['ci'][0]:.0%}-{r['ci'][1]:.0%}), placement "
          f"{r['place_mm_success']} mm, time {r['time_s']} s, {r['outcomes']}; feedback cut {RANDS[name]['cut']['summary']['outcomes']}")
    del rnet; gc.collect()
    print(f"peak memory {peak_mem_gb():.2f} GB")

## Step 3: the final comparison (deliverable), video, results file

In [ ]:
SUMM = {'connectome, C4 (week 10)': CONN_T['summary'], **{k: v['summary'] for k, v in RANDS.items()}}
TABLE = pd.DataFrame(comparison_rows(SUMM)).set_index('policy')
TABLE['w_scale'] = [W] + [v.get('w_scale') for v in RANDS.values()]
display(TABLE.style.format({c: '{:.0%}' for c in ('success rate', 'lifted', 'carried to zone', 'stalled')}, na_rep='-'))
display(pd.DataFrame({k: s['outcomes'] for k, s in SUMM.items()}).fillna(0).astype(int).T)
CMP = compare10b(CONN_T, RANDS)
LINES = describe10b(CMP)
for line in LINES:
    print(line)

VIDEO = None
EPISODES = [e for k, v in RANDS.items() if not k.endswith('(week 10)') for e in v['video_eps']]
for e in EPISODES:
    print(f"video: {e['net']}, seed {e['seed']}: {e['outcome']}")
if EPISODES:
    pickle.dump(EPISODES, open(OUT / 'week10b_episodes.pkl', 'wb'))
    from sim.video7 import render
    VIDEO = OUT / f'week10b_shuffles_{HOST}.mp4'
    if render(str(OUT / 'week10b_episodes.pkl'), str(VIDEO), slow=0.5):
        from IPython.display import Video
        display(Video(str(VIDEO), embed=True, width=900))
    else:
        VIDEO = None

summary = dict(host=HOST, quick=QUICK, gpu=GPU, test_seeds=[CFG['test_seeds'][0], CFG['test_seeds'][-1]],
               target_frac_firing=TARGET, match={str(k): {kk: v for kk, v in m.items() if kk != 'tries'} for k, m in MATCH.items()},
               w_scale_week7_seed0=W_RAND0, table=TABLE.reset_index().to_dict('records'),
               outcomes={k: s['outcomes'] for k, s in SUMM.items()}, comparison=CMP, lines=LINES,
               networks={k: dict(w_scale=v.get('w_scale'), dagger_log=v['dagger']['log'], cut=v['cut']['summary'],
                                 hover=v.get('hover'), trap=v.get('trap'), grip_agreement=v.get('grip_agreement'),
                                 pool=len(v['pool_cols']), per_seed={o['seed']: o['outcome'] for o in v['outs']},
                                 place_mm={o['seed']: round(o['place_error_mm'], 1) for o in v['outs']},
                                 wall_min=v.get('wall_min')) for k, v in RANDS.items()},
               connectome_per_seed={o['seed']: o['outcome'] for o in CONN_T['outs']},
               connectome_place_mm={o['seed']: round(o['place_error_mm'], 1) for o in CONN_T['outs']},
               wall_min=(time.perf_counter() - T_START) / 60)
(OUT / f'week10b_results_{HOST}.json').write_text(json.dumps(summary, indent=1, default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o)))
print(f"this session {summary['wall_min']:.0f} min. Send me week10b_results_{HOST}.json"
      f"{' and the video' if VIDEO else ''} from {OUT}.")